# Cython Magic Demo
Quick test of `%%cython` cell magic in Jupyter.

In [2]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


Compiler used:

In [5]:
import setuptools._distutils.ccompiler as ccompiler
print(ccompiler.get_default_compiler())

msvc


In [4]:
%%cython --verbose

def fib_cy(int n):
    cdef int a = 0, b = 1, i
    for i in range(n):
        a, b = b, a + b
    return a

[1/1] Cythonizing C:\Users\khazy\.ipython\cython\_cython_magic_e1714406fbdda19f396d6b4e9774a3a613f1a2d380f53cd36c478fca7c2752ed.pyx
Content of stdout:
_cython_magic_e1714406fbdda19f396d6b4e9774a3a613f1a2d380f53cd36c478fca7c2752ed.c
   Creating library C:\Users\khazy\.ipython\cython\Users\khazy\.ipython\cython\_cython_magic_e1714406fbdda19f396d6b4e9774a3a613f1a2d380f53cd36c478fca7c2752ed.cp314-win_amd64.lib and object C:\Users\khazy\.ipython\cython\Users\khazy\.ipython\cython\_cython_magic_e1714406fbdda19f396d6b4e9774a3a613f1a2d380f53cd36c478fca7c2752ed.cp314-win_amd64.exp
Generating code
Finished generating code

In [6]:
# Pure Python version for comparison
def fib_py(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

In [7]:
# Verify both give the same result
print(f"Cython: {fib_cy(30)}")
print(f"Python: {fib_py(30)}")

Cython: 832040
Python: 832040


In [8]:
# Benchmark
%timeit fib_cy(1000)
%timeit fib_py(1000)

464 ns ± 15.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
56.6 μs ± 1.47 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


# Profiling Workflow: Python → Profile → Cython

The goal: start with a pure Python function that uses numpy,
profile it to find hotspots, then use Cython static typing
to speed up just the slow parts.

In [12]:
import numpy as np

In [13]:
def slow_moving_average(data, window_size):
    """Pure Python moving average - deliberately not using np.convolve
    so we can see the loop overhead."""
    n = len(data)
    result = np.empty(n - window_size + 1)
    for i in range(n - window_size + 1):
        total = 0.0
        for j in range(window_size):
            total += data[i + j]
        result[i] = total / window_size
    return result

# Test data
data = np.random.randn(50_000)
window = 100

# Verify it works
result = slow_moving_average(data, window)
print(f"Result shape: {result.shape}, first 5 values: {result[:5]}")

Result shape: (49901,), first 5 values: [0.22202726 0.22171979 0.2055094  0.18202451 0.18164216]


## Step 1: Baseline timing

In [14]:
%timeit slow_moving_average(data, window)

794 ms ± 32.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Step 2: Profile with cProfile to find hotspots
`%prun` shows where time is spent at the function-call level.

In [15]:
%prun -s cumulative slow_moving_average(data, window)

         80 function calls in 0.782 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.767    0.767 {built-in method builtins.exec}
        1    0.000    0.000    0.767    0.767 <string>:1(<module>)
        1    0.766    0.766    0.766    0.766 284397751.py:1(slow_moving_average)
        2    0.015    0.007    0.015    0.007 {method '__exit__' of 'sqlite3.Connection' objects}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        2    0.000    0.000    0.000    0.000 traitlets.py:708(__set__)
        2    0.000    0.000    0.000    0.000 traitlets.py:3631(set)
        2    0.000    0.000    0.000    0.000 traitlets.py:689(set)
        2    0.000    0.000    0.000    0.000 traitlets.py:718(_validate)
        1    0.000    0.000    0.000    0.000 traitlets.py:1512(_notify_trait)
        2    0.000    0.000    0.000    0.000 traitlets.py:3474(valid

## Step 3: Line-level profiling (optional, needs line_profiler)
`%lprun` shows time spent per line — the most useful view for deciding what to type.
Install with `pip install line_profiler` if not already available.

In [ ]:
%load_ext line_profiler
%lprun -f slow_moving_average slow_moving_average(data, window)

## Step 4: Cython annotated output
`%%cython -a` produces an HTML annotation. Yellow lines = Python overhead (slow).
White lines = pure C (fast). This tells you exactly which lines need static typing.

In [ ]:
%%cython -a

import numpy as np
cimport numpy as cnp

# No static types yet — just a direct port to Cython.
# The annotation will show lots of yellow (Python interaction).
def moving_average_v1(cnp.ndarray[double] data, int window_size):
    cdef int n = len(data)
    cdef cnp.ndarray[double] result = np.empty(n - window_size + 1)
    for i in range(n - window_size + 1):
        total = 0.0
        for j in range(window_size):
            total += data[i + j]
        result[i] = total / window_size
    return result

## Step 5: Fully typed Cython version
Based on profiling, the inner loops over `i` and `j` and the array accesses
are the hotspots. We add `cdef int i, j` and `cdef double total` to eliminate
Python object overhead in the tight loop. We also disable bounds checking.

In [ ]:
%%cython -a
# cython: boundscheck=False, wraparound=False

import numpy as np
cimport numpy as cnp

def moving_average_v2(cnp.ndarray[double] data, int window_size):
    cdef int n = len(data)
    cdef int i, j
    cdef double total
    cdef cnp.ndarray[double] result = np.empty(n - window_size + 1)
    for i in range(n - window_size + 1):
        total = 0.0
        for j in range(window_size):
            total += data[i + j]
        result[i] = total / window_size
    return result

## Step 6: Compare all versions
Run the same data through all three and compare timings.

In [ ]:
# Verify all produce the same result
r0 = slow_moving_average(data, window)
r1 = moving_average_v1(data, window)
r2 = moving_average_v2(data, window)
print(f"v1 matches: {np.allclose(r0, r1)}")
print(f"v2 matches: {np.allclose(r0, r2)}")

In [ ]:
print("Pure Python:")
%timeit slow_moving_average(data, window)

print("\nCython v1 (typed arrays, untyped loop vars):")
%timeit moving_average_v1(data, window)

print("\nCython v2 (fully typed + boundscheck off):")
%timeit moving_average_v2(data, window)

print("\nNumPy reference (np.convolve):")
%timeit np.convolve(data, np.ones(window)/window, mode='valid')